# Methanol Decomposition Reactor — Physics Digital Twin

**Physics model** with LHHW-type kinetics for methanol decomposition.

| Role | Variables |
|------|-----------|
| **Inputs (4)** | Inlet Temperature (°C), Inlet Pressure (bar_g), **Heating media temperature (°C)**, **Methanol** molar flow rate (kgmole/h) |
| **Outputs (8)** | Temp_0.2m … Temp_1m (°C), Outlet Pressure (bar_g), Total heat input (kW), Overall Conversion (%) |

**Reaction:** CH₃OH → CO + 2H₂

**Feed = methanol only** (no steam / water in this model).

**Heating media temperature `T_heat_C` is an input per case** (held constant along the bed).

**Wall heat transfer:** overall U = 1 / (1/h_o + 1/h_i) is calculated first, then used as h_heat.

## 1. Imports

In [ ]:
from pathlib import Path
# Import Path to work with file and folder locations on disk

import matplotlib.pyplot as plt
# Import matplotlib for making plots and graphs

import numpy as np
# Import NumPy for fast math and array calculations

import pandas as pd
# Import pandas for reading CSV and working with tables

from scipy.integrate import solve_ivp
# Import ODE solver used to integrate the reactor model along length

DATA_FILE = Path("Methanol Reforming Data.xlsx")
# Store the CSV file path used for validation data

MEASUREMENT_POSITIONS_M = np.array([0.2, 0.4, 0.6, 0.8, 1.0])
# Axial positions (m) where temperatures are reported in the data

P_ATM_BAR = 1.01325
# Standard atmospheric pressure in bar (needed for bar_g ↔ absolute conversions)

# Heating media temperature T_heat_C is a per-case plant input (from Methanol Reforming Data.xlsx)

## 2. Load data — Methanol Reforming Data.xlsx

Operating cases from **Methanol Reforming Data.xlsx** (4 inputs + 8 reference outputs). Reactor geometry is **fixed** (Section 3).

- **Total heat input:** sum of Excel segment heat flows (q1…q5); Excel is negative for heat in → sign flipped so heat input is positive.
- **Overall conversion:** Excel `Conversion at 1m (X-5)` stores a fraction → converted to %.

In [ ]:
def load_data(path: Path):
    # Load and clean operating cases from Methanol Reforming Data.xlsx
    raw = pd.read_excel(path, sheet_name=0, header=0, skiprows=[1])  # row1 = units
    # raw = Excel first sheet with original column names (Ti, Th, T1, …)

    data = raw.rename(
        columns={
            "Inlet Temperature (Ti)": "T_in_C",
            # T_in_C = inlet gas temperature [°C]

            "Inlet Pressure (Pi)": "P_in_bar_g",
            # P_in_bar_g = inlet gauge pressure [bar_g]

            "Heating media Temp (Th)": "T_heat_C",
            # T_heat_C = heating media temperature for this case [°C]

            "Inlet molar flow rate (mi)": "F_in_kgmole_h",
            # F_in_kgmole_h = inlet methanol molar flow [kgmole/h]

            "Temperature at 0.2m (T1)": "T_0.2m_C",
            # T_0.2m_C = temperature at 0.2 m along bed [°C]

            "Temperature at 0.4m (T2)": "T_0.4m_C",
            # T_0.4m_C = temperature at 0.4 m [°C]

            "Temperature at 0.6m (T3)": "T_0.6m_C",
            # T_0.6m_C = temperature at 0.6 m [°C]

            "Temperature at 0.8m (T4)": "T_0.8m_C",
            # T_0.8m_C = temperature at 0.8 m [°C]

            "Temperature at 1m (T5)": "T_1m_C",
            # T_1m_C = outlet temperature at 1.0 m [°C]

            "Pressure at 1m (P5)": "P_out_bar_g",
            # P_out_bar_g = outlet gauge pressure [bar_g]
        }
    )

    # Ensure numeric dtypes after dropping the Excel units row
    for _col in [
        "T_in_C", "P_in_bar_g", "T_heat_C", "F_in_kgmole_h",
        "T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C", "P_out_bar_g",
        "Conversion at 1m (X-5)",
    ]:
        if _col in data.columns:
            data[_col] = pd.to_numeric(data[_col], errors="coerce")
    for _col in [c for c in data.columns if str(c).startswith("Heat flow")]:
        data[_col] = pd.to_numeric(data[_col], errors="coerce")

    heat_cols = [c for c in data.columns if str(c).startswith("Heat flow")]
    # heat_cols = the 5 segment heat-flow columns q1 … q5 [kW]

    data["Q_total_kW"] = -data[heat_cols].sum(axis=1)
    # Flip sign and sum: Excel reports heat INTO the reactor as negative; model uses positive [kW]

    data["X_overall_pct"] = data["Conversion at 1m (X-5)"] * 100.0
    # Excel stores conversion as a fraction → convert to percentage [%]


    ok = (
        data["X_overall_pct"].between(0, 100)
        # Filter: conversion must be a physical percentage

        & data["Q_total_kW"].between(-50, 50)
        # Filter: heat input between -50 and 50 kW (sanity check)

        & data[["T_in_C", "P_in_bar_g", "T_heat_C", "F_in_kgmole_h"]].notna().all(axis=1)
        # Filter: all 4 inputs must be present (not missing)
    )
    # ok = boolean mask for valid simulation rows

    print(f"Total rows in file          : {len(raw)}")

    data = data.loc[ok].reset_index(drop=True)
    # Apply filter and reset row index to 0, 1, 2, ...

    return data
    # Return cleaned dataframe for validation
df = load_data(DATA_FILE)
# df = cleaned dataset from Methanol Reforming Data.xlsx

INPUT_COLS = ["T_in_C", "P_in_bar_g", "T_heat_C", "F_in_kgmole_h"]
# INPUT_COLS = list of 4 model input column names

OUTPUT_COLS = [
    "T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C",
    "P_out_bar_g", "Q_total_kW", "X_overall_pct",
]
# OUTPUT_COLS = list of 8 model output column names

print(f"Valid rows                  : {len(df)}")
# Print count of rows after filtering

display(df[INPUT_COLS + OUTPUT_COLS].head())
# Show first 5 rows (inputs + reference outputs) for quick inspection

## 3. Fixed reactor geometry & kinetic parameters

**Reaction:** CH₃OH → CO + 2H₂ | **Reaction heat ΔH(25°C) = 2.2e4 kcal/kgmole** (SI conversion; temperature-dependent via ΔCp)

**Kinetics:** R = 1.987 kcal/(kgmole·K) with activation energies in kcal/kgmole; concentrations in kgmole/m³, rate in kgmole/(m³·s) (converted to mol/(m³·s) for the balances).

**Fixed geometry (always these values):**

| Parameter | Value | Unit |
|-----------|-------|------|
| Reactor diameter | 0.1 | m |
| Reactor length | 1.0 | m |
| Heat medium side HTC (h_o) | 115 | W/m²/°C |
| Heat medium side temperature | input `T_heat_C` | °C |
| Catalyst bed void fraction | 0.33 | — |
| Reaction heat | 2.2e4 | kcal/kgmole |
| Solid heat capacity | 2.5 | kJ/kg/°C |

**Wall heat transfer — U first, then h_heat:**

- Bed side (Specchia correlation): Nu_w = h_i·dp/k_gas = 2·ε + 0.0835 · Re_p^0.91, with Re_p = dp·G/μ (the 2·ε term is the stagnant-bed conduction contribution)
- Overall: **U = 1 / (1/h_o + 1/h_i)**
- **h_heat = U** → wall heat flux q = h_heat · πD · (T_heat − T)

G (mass flux) is constant along the bed, so U is calculated once per case from the inlet methanol flow.

Solid heat capacity only affects transient (time-dependent) behaviour; it has no effect on this steady-state model.

In [ ]:
# ============================================================
# 1. UNIVERSAL GAS CONSTANT
# ============================================================
R_kcal = 1.987          # kcal/(kgmole·K)
# R_kcal = universal gas constant used in Arrhenius terms (E in kcal/kgmole)

KCAL_PER_KGMOLE_TO_J_PER_MOL = 4184.0 / 1000.0
R = R_kcal * KCAL_PER_KGMOLE_TO_J_PER_MOL  # ~8.314 J/(mol·K)
# R = same constant in SI; used only in the ideal gas law (concentration, density)

# ============================================================
# 2. FIXED REACTOR PARAMETERS
# ============================================================
D = 0.1                  # Reactor diameter [m]
# D = inner tube diameter of the catalyst bed

L = 1                  # Reactor length [m]
# L = axial length of reactor (integration runs from 0 to L)

h_o = 115.0              # Heat medium side HTC [W/m2.C]
# h_o = heat transfer coefficient on the heating-jacket side of the wall

# T_heat_C is a per-case input passed into reactor_model / predict_outputs

epsilon = 0.33           # Catalyst bed void fraction [-]
# epsilon = fraction of bed volume that is gas (not solid catalyst)

Cp_solid = 2.5e3         # Solid heat capacity [J/(kg·°C)] = 2.5 kJ/kg-°C
# Cp_solid = catalyst heat capacity; only matters for transients, unused at steady state

dp = 1.000e-3            # Catalyst particle diameter [m]
# dp = catalyst pellet size; used in Ergun pressure drop and bed-side HTC

mu_gas = 2.0e-5          # Gas viscosity [Pa·s]
# mu_gas = dynamic gas viscosity for Ergun equation and Reynolds number

k_gas = 0.035            # Gas thermal conductivity [W/(m·K)]
# k_gas = gas conductivity used in bed-side Nusselt number (methanol vapour ~0.03–0.04)

Ac = np.pi * D**2 / 4    # Reactor cross-sectional area [m²]
# Ac = flow area inside the circular tube (π·D²/4)

perimeter = np.pi * D    # Heat-transfer perimeter [m]
# perimeter = wetted wall length per meter of bed for heat transfer

# ============================================================
# 3. MOLECULAR WEIGHTS
# ============================================================
MW_CH3OH = 32.04e-3      # kg/mol
# MW_CH3OH = methanol molecular weight for mass-flow calculations

MW_CO    = 28.01e-3      # kg/mol
# MW_CO = carbon monoxide molecular weight

MW_H2    = 2.016e-3      # kg/mol
# MW_H2 = hydrogen molecular weight

# ============================================================
# 4. HEAT CAPACITIES
# ============================================================
Cp_CH3OH = 43.0          # J/mol.K
# Cp_CH3OH = molar heat capacity of methanol (energy balance)

Cp_CO    = 29.0          # J/mol.K
# Cp_CO = molar heat capacity of CO

Cp_H2    = 29.0          # J/mol.K
# Cp_H2 = molar heat capacity of hydrogen

# ============================================================
# 5. WALL HEAT TRANSFER — calculate U first, then h_heat = U
# ============================================================
def bed_side_htc(F_in_mol_s):
    # Bed-side wall HTC h_i [W/(m²·°C)] from Specchia correlation
    mass_flux = F_in_mol_s * MW_CH3OH / Ac
    # mass_flux = G [kg/(m²·s)]; constant along the bed (mass is conserved)

    Re_p = dp * mass_flux / mu_gas
    # Re_p = particle Reynolds number

    Nu_w = 2.0 * epsilon + 0.0835 * Re_p**0.91
    # Nu_w = wall Nusselt number based on particle diameter (Specchia)

    return Nu_w * k_gas / dp
    # h_i = Nu_w · k_gas / dp


def overall_U(F_in_mol_s):
    # Overall wall heat transfer coefficient U [W/(m²·°C)]
    h_i = bed_side_htc(F_in_mol_s)
    # h_i = bed (process) side coefficient

    return 1.0 / (1.0 / h_o + 1.0 / h_i)
    # U = 1 / (1/h_o + 1/h_i)  — series resistances, thin wall neglected


# ============================================================
# 6. REACTION KINETICS
# r = [7e10 exp(-19120/RT) C_CH3OH] / [1 + 1e7 exp(-6036.8/RT) C_CO]
# R in kcal/(kgmole·K), E in kcal/kgmole, C in kgmole/m³, r in kgmole/(m³·s)
# ============================================================
A_kinetic = 7.0000e10
# A_kinetic = pre-exponential factor for main decomposition rate

E_kinetic = 19120.0      # kcal/kgmole
# E_kinetic = activation energy for methanol reaction [kcal/kgmole]

A_CO = 1.0000e7
# A_CO = pre-exponential factor for CO adsorption in LHHW denominator

E_CO = 6036.8            # kcal/kgmole
# E_CO = activation energy for CO adsorption term [kcal/kgmole]

# ============================================================
# 7. REACTION HEAT  —  CH3OH -> CO + 2H2
# ============================================================
# Given as ΔH(25°C) = 2.2e4 kcal/kgmole, convert to SI J/mol
delta_H_ref_kcal = 2.2e4
T_ref_C = 25.0
delta_H_ref = delta_H_ref_kcal * KCAL_PER_KGMOLE_TO_J_PER_MOL  # ~9.205e4 J/mol
delta_Cp = Cp_CO + 2.0 * Cp_H2 - Cp_CH3OH  # J/(mol·K)


def reaction_enthalpy(T_C):
    # ΔH(T) = ΔH(25°C) + ΔCp·(T − 25)  [J/mol]
    return delta_H_ref + delta_Cp * (float(T_C) - T_ref_C)


delta_H = reaction_enthalpy(T_ref_C)  # reference value used in prints
# delta_H / reaction_enthalpy = endothermic heat of reaction [J/mol]

print("Fixed reactor parameters:")
# Print summary header

print(f"  D        = {D} m")
# Show reactor diameter

print(f"  L        = {L} m")
# Show reactor length

print(f"  h_o      = {h_o} W/m²-K (heat medium side)")
# Show heat medium side heat transfer coefficient

print("  T_heat   = per-case input (T_heat_C)")
# Show heating medium temperature

print(f"  epsilon  = {epsilon}")
# Show bed void fraction

print(f"  Cp_solid = {Cp_solid / 1e3} kJ/kg-°C")
# Show solid heat capacity

print(f"  R        = {R_kcal} kcal/(kgmole·K) -> {R:.6f} J/(mol·K)")
print(f"  delta_H(25C) = {delta_H_ref_kcal:.3e} kcal/kgmole -> {delta_H_ref:.3e} J/mol")
print(f"  delta_H(350C example) = {reaction_enthalpy(350):.3e} J/mol")
# Show reaction enthalpy in J/mol

print("\nWall heat transfer (U first, then h_heat = U):")
for F_kgmole_h in sorted(df["F_in_kgmole_h"].unique()):
    F_mol_s = F_kgmole_h * 1000.0 / 3600.0
    print(f"  F = {F_kgmole_h:5.2f} kgmole/h : h_i = {bed_side_htc(F_mol_s):7.2f}, "
          f"U = h_heat = {overall_U(F_mol_s):7.2f} W/m²-K")
# Show bed-side h_i and overall U for each methanol flow in the data

## 4. Unit conversions (Excel inputs → model units)

In [ ]:
def bar_g_to_Pa(P_bar_g):
    # P_bar_g = inlet/outlet gauge pressure from Excel [bar_g]
    """Convert gauge pressure (bar_g) to absolute pressure [Pa]."""
    return (P_bar_g + P_ATM_BAR) * 1.0e5
    # Absolute pressure [Pa] = (gauge + atmospheric) converted bar → Pa


def Pa_to_bar_g(P_Pa):
    # P_Pa = absolute pressure from ODE model [Pa]
    """Convert absolute pressure [Pa] to gauge pressure (bar_g)."""
    return P_Pa / 1.0e5 - P_ATM_BAR
    # bar_g = absolute Pa → bar, then subtract atmospheric bar


def kgmole_h_to_mol_s(F_kgmole_h):
    # F_kgmole_h = methanol molar flow from Excel [kgmole/h]
    """Convert methanol molar flow from kgmole/h (Excel) to mol/s (model)."""
    return F_kgmole_h * 1000.0 / 3600.0
    # mol/s = kgmole/h × 1000 mol/kgmole ÷ 3600 s/h

## 5. Physics model — reaction rate & reactor ODEs

**All temperatures in °C** (Kelvin used only inside Arrhenius / ideal-gas formulas).

Rate law evaluated on a kgmole basis (C in kgmole/m³, R in kcal/(kgmole·K)), then converted to mol/(m³·s).

State vector: `[F_CH3OH, F_CO, F_H2, T(°C), P(Pa), Q(W)]`

Reaction pathway: **CH₃OH → CO + 2H₂**

Wall heat flux uses **h_heat = U** (per case) and **T_heat_C** (plant input, constant along z).

**Conversion:** calculated as decimal internally, reported as **percentage (%)** in all outputs.

In [ ]:
def reaction_rate(T_C, P, F_CH3OH, F_CO, F_H2):
    # Compute LHHW reaction rate [mol/(m³·s)] from T, P, and species flows
    """Rate in mol/(m³·s). T_C in °C; Kelvin used only inside Arrhenius / gas-law terms."""
    T_abs = T_C + 273.15
    # T_abs = absolute temperature [K] for Arrhenius and gas law

    F_total = F_CH3OH + F_CO + F_H2
    # F_total = total molar flow of all gas species at this position [mol/s]

    F_total = max(F_total, 1e-12)
    # Prevent division by zero when flow is extremely small

    y_CH3OH = F_CH3OH / F_total
    # y_CH3OH = methanol mole fraction in the gas mixture

    y_CO = F_CO / F_total
    # y_CO = CO mole fraction (used in adsorption denominator)

    C_total = P / (R * T_abs) / 1000.0
    # C_total = total molar concentration from ideal gas law [kgmole/m³]

    C_CH3OH = y_CH3OH * C_total
    # C_CH3OH = methanol concentration driving the forward reaction [kgmole/m³]

    C_CO = y_CO * C_total
    # C_CO = CO concentration for Langmuir inhibition term [kgmole/m³]

    numerator = (
        A_kinetic
        * np.exp(-E_kinetic / (R_kcal * T_abs))
        * C_CH3OH
    )
    # numerator = Arrhenius rate × methanol concentration (R in kcal/(kgmole·K))

    denominator = (
        1.0
        + A_CO * np.exp(-E_CO / (R_kcal * T_abs)) * C_CO
    )
    # denominator = 1 + CO adsorption inhibition (LHHW form)

    r_kgmole = numerator / denominator
    # r_kgmole = volumetric reaction rate [kgmole/(m³·s)]

    return r_kgmole * 1000.0
    # Return volumetric reaction rate r [mol/(m³·s)]


def reactor_model(z, y, h_heat, T_heat_C):
    # T_heat_C = heating media temperature for this case [C], constant along z
    # ODE system: how state variables change along reactor length z [m]
    # h_heat = overall wall HTC U for this case [W/(m²·°C)]
    F_CH3OH = max(y[0], 1e-12)
    # F_CH3OH = methanol molar flow [mol/s]; state index 0

    F_CO = max(y[1], 0.0)
    # F_CO = CO molar flow [mol/s]; state index 1

    F_H2 = max(y[2], 0.0)
    # F_H2 = hydrogen molar flow [mol/s]; state index 2

    T_C = y[3]
    # T_C = gas temperature [°C]; state index 3

    P = max(y[4], 1.0e3)
    # P = absolute gas pressure [Pa]; state index 4; floor avoids zero pressure

    Q = y[5]
    # Q = cumulative heat transferred into bed [W]; state index 5

    F_total = F_CH3OH + F_CO + F_H2
    # F_total = sum of all species flows at this axial position

    r = reaction_rate(T_C, P, F_CH3OH, F_CO, F_H2)
    # r = local volumetric reaction rate [mol/(m³·s)]

    reaction_per_length = r * Ac
    # reaction_per_length = molar reaction rate per meter of bed [mol/(s·m)]

    # CH3OH -> CO + 2H2 stoichiometry
    dF_CH3OH_dz = -reaction_per_length
    # Methanol is consumed along the bed

    dF_CO_dz = reaction_per_length
    # CO is produced 1 mol per mol CH3OH reacted

    dF_H2_dz = 2.0 * reaction_per_length
    # H2 is produced 2 mol per mol CH3OH reacted

    q_transfer = h_heat * perimeter * (T_heat_C - T_C)
    # q_transfer = wall heat flux into gas [W/m] (positive when T_C < T_heat_C), h_heat = U

    Cp_flow = (
        F_CH3OH * Cp_CH3OH
        + F_CO * Cp_CO
        + F_H2 * Cp_H2
    )
    # Cp_flow = total heat capacity flow of gas stream [J/(s·K)]

    Cp_flow = max(Cp_flow, 1e-9)
    # Avoid division by zero in temperature ODE

    q_reaction = reaction_enthalpy(T_C) * reaction_per_length
    # q_reaction = heat absorbed by endothermic reaction [W/m]
    # uses ΔH(T) = ΔH(25°C) + ΔCp·(T − 25)

    dT_dz = (q_transfer - q_reaction) / Cp_flow
    # dT_dz = temperature gradient along bed [°C/m]

    mass_flow = (
        F_CH3OH * MW_CH3OH
        + F_CO * MW_CO
        + F_H2 * MW_H2
    )
    # mass_flow = total gas mass flow rate [kg/s]

    MW_mix = mass_flow / max(F_total, 1e-12)
    # MW_mix = average molecular weight of gas mixture [kg/mol]

    T_abs = T_C + 273.15
    # T_abs = Kelvin temperature for density calculation only

    rho = P * MW_mix / (R * T_abs)
    # rho = gas density from ideal gas law [kg/m³]

    velocity = mass_flow / max(rho * Ac, 1e-12)
    # velocity = superficial gas velocity through bed [m/s]

    viscous_term = (
        150.0 * mu_gas * (1.0 - epsilon) ** 2
        / (epsilon ** 3 * dp ** 2) * velocity
    )
    # viscous_term = viscous contribution to Ergun pressure gradient [Pa/m]

    inertial_term = (
        1.75 * rho * (1.0 - epsilon)
        / (epsilon ** 3 * dp) * velocity ** 2
    )
    # inertial_term = inertial contribution to Ergun pressure gradient [Pa/m]

    dP_dz = -(viscous_term + inertial_term)
    # dP_dz = pressure drop along bed [Pa/m] (negative = pressure decreases)

    dQ_dz = q_transfer
    # dQ_dz = rate of heat accumulation along bed [W/m]

    return [
        dF_CH3OH_dz,
        dF_CO_dz,
        dF_H2_dz,
        dT_dz,
        dP_dz,
        dQ_dz,
    ]
    # Return all 6 ODE derivatives for the solver

## 6. Digital twin — 4 inputs → 8 outputs

In [ ]:
def predict_outputs(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h, n_points=1000):
    # Digital twin: 4 plant inputs → 8 outputs (+ profiles and outlet flows)
    """
    INPUTS (plant / Excel units):
        T_in_C           — Inlet Temperature [°C]
        P_in_bar_g       — Inlet Pressure [bar_g]
        T_heat_C         — Heating media temperature [C] (constant along reactor)
        F_in_kgmole_h    — Inlet methanol molar flow rate [kgmole/h]

    OUTPUTS:
        T_0.2m_C … T_1m_C  — axial temperatures [°C]
        P_out_bar_g        — outlet pressure [bar_g]
        Q_total_kW         — total heat input [kW]
        X_overall_pct      — overall methanol conversion [%]
    """
    F_CH3OH_in = kgmole_h_to_mol_s(F_in_kgmole_h)
    # F_CH3OH_in = inlet methanol flow converted to [mol/s]

    P_in_Pa = bar_g_to_Pa(P_in_bar_g)
    # P_in_Pa = inlet absolute pressure [Pa] for the ODE model

    U = overall_U(F_CH3OH_in)
    # U = overall wall HTC from h_o and bed-side h_i [W/(m²·°C)]

    h_heat = U
    # h_heat = U, used in the wall heat flux of the ODE model

    y0 = [
        F_CH3OH_in,
        0.0,       # F_CO_in
        0.0,       # F_H2_in
        T_in_C,
        P_in_Pa,
        0.0,       # Q
    ]
    # y0 = initial state at z=0: [F_CH3OH, F_CO, F_H2, T_C, P, Q]

    z_eval = np.linspace(0.0, L, n_points)
    # z_eval = axial grid points from inlet (0) to outlet (L) [m]

    sol = solve_ivp(
        reactor_model,
        (0.0, L),
        y0,
        t_eval=z_eval,
        method="BDF",
        rtol=1e-6,
        atol=1e-9,
        args=(h_heat, float(T_heat_C)),
    )
    # Integrate ODEs along bed; BDF handles stiff kinetics; h_heat passed to ODEs

    if not sol.success:
        raise RuntimeError(sol.message)
    # Raise error if ODE solver failed

    F_CH3OH_out = sol.y[0, -1]
    # F_CH3OH_out = methanol flow at reactor outlet [mol/s]

    T_profile_C = sol.y[3]
    # T_profile_C = full axial temperature profile [°C] along z

    P_out_Pa = sol.y[4, -1]
    # P_out_Pa = outlet absolute pressure [Pa]

    Q_total_W = sol.y[5, -1]
    # Q_total_W = total heat transferred into bed at outlet [W]

    T_axial_C = np.interp(MEASUREMENT_POSITIONS_M, sol.t, T_profile_C)
    # T_axial_C = temperatures at Excel positions (0.2, 0.4, …, 1.0 m)

    X_overall_pct = (F_CH3OH_in - F_CH3OH_out) / F_CH3OH_in * 100.0
    # X_overall_pct = methanol conversion as percentage [%]

    Q_kW = Q_total_W / 1000.0
    # Q_kW = total heat input reported in kilowatts [kW]

    return {
        "T_0.2m_C": float(T_axial_C[0]),
        # T at z = 0.2 m [°C]

        "T_0.4m_C": float(T_axial_C[1]),
        # T at z = 0.4 m [°C]

        "T_0.6m_C": float(T_axial_C[2]),
        # T at z = 0.6 m [°C]

        "T_0.8m_C": float(T_axial_C[3]),
        # T at z = 0.8 m [°C]

        "T_1m_C": float(T_axial_C[4]),
        # Outlet temperature at z = 1.0 m [°C]

        "P_out_bar_g": float(Pa_to_bar_g(P_out_Pa)),
        # Outlet gauge pressure converted from Pa to bar_g

        "Q_total_kW": float(Q_kW),
        # Total heat transferred into bed [kW]

        "X_overall_pct": float(X_overall_pct),
        # Methanol conversion [%]

        "z_m": sol.t,
        # Axial positions along full integration grid [m]

        "T_profile_C": T_profile_C,
        # Full axial temperature profile [°C] at each z_m point

        "F_CH3OH_out": float(F_CH3OH_out),
        # Methanol molar flow at outlet [mol/s]

        "F_CO_out": float(sol.y[1, -1]),
        # CO molar flow at outlet [mol/s]

        "F_H2_out": float(sol.y[2, -1]),
        # H2 molar flow at outlet [mol/s]

        "h_heat_W_m2K": float(h_heat),
        # Overall wall HTC U used as h_heat [W/(m²·°C)]
    }

## 7. Example run — single operating point

In [ ]:
# Example = first cleaned Excel row (so T_heat_C comes from Excel, not a fixed value)
ex = df.iloc[0]
T_in_C = float(ex["T_in_C"])
P_in_bar_g = float(ex["P_in_bar_g"])
T_heat_C = float(ex["T_heat_C"])  # heating media temp from Excel for this case
F_in_kgmole_h = float(ex["F_in_kgmole_h"])

pred = predict_outputs(T_in_C, P_in_bar_g, T_heat_C, F_in_kgmole_h)
# pred = dict of all model outputs for this operating point

print("\n" + "=" * 55)
# Print section divider

print("      METHANOL DECOMPOSITION PHYSICS MODEL")
# Print model title

print("=" * 55)
# Print closing divider

print("\nREACTION:")
# Print reaction section header

print("CH3OH -> CO + 2H2")
# Show stoichiometry being modeled

print("\nINPUTS:")
# Print inputs section header

print(f"  Inlet Temperature     : {T_in_C:.2f} °C")
# Display inlet temperature

print(f"  Inlet Pressure        : {P_in_bar_g:.3f} bar_g")
# Display inlet gauge pressure

print(f"  Inlet methanol flow   : {F_in_kgmole_h:.3f} kgmole/h")
# Display methanol feed flow (only species in feed)

print(f"  Heating media temp    : {T_heat_C:.2f} °C")
# Display heating medium temperature

print(f"  U = h_heat            : {pred['h_heat_W_m2K']:.2f} W/m²-K")
# Display overall wall HTC used for this case

print("\nAXIAL TEMPERATURE PROFILE:")
# Print axial temperature section header

for z, key in zip(MEASUREMENT_POSITIONS_M, [
    "T_0.2m_C", "T_0.4m_C", "T_0.6m_C", "T_0.8m_C", "T_1m_C"
]):
    # Loop over measurement positions and matching output keys
    print(f"  Temp_{z:g}m = {pred[key]:.2f} °C")
    # Print temperature at each axial location [°C]

print("\nOUTLET RESULTS:")
# Print outlet results section header

print(f"  Outlet temperature    : {pred['T_1m_C']:.2f} °C")
# Outlet gas temperature at z = 1.0 m

print(f"  Outlet pressure       : {pred['P_out_bar_g']:.4f} bar_g")
# Outlet gauge pressure

print(f"  Total heat input      : {pred['Q_total_kW']:.4f} kW")
# Total heat transferred into the bed [kW]

print(f"  Overall conversion    : {pred['X_overall_pct']:.2f} %")
# Methanol conversion as percentage [%]

print("\nOUTLET MOLAR FLOWS:")
# Print outlet species flows section

print(f"  CH3OH = {pred['F_CH3OH_out']:.6e} mol/s")
# Remaining methanol flow at outlet [mol/s]

print(f"  CO    = {pred['F_CO_out']:.6e} mol/s")
# CO produced at outlet [mol/s]

print(f"  H2    = {pred['F_H2_out']:.6e} mol/s")
# H2 produced at outlet [mol/s]

print("=" * 55)
# Print final divider

fig, ax = plt.subplots(figsize=(7, 4))
# Create figure and axes for temperature profile plot

ax.plot(pred["z_m"], pred["T_profile_C"], lw=2, label="Physics model")
# Plot full axial temperature profile; z_m = position [m], T_profile_C = T [°C]

ax.scatter(
    MEASUREMENT_POSITIONS_M,
    [pred["T_0.2m_C"], pred["T_0.4m_C"], pred["T_0.6m_C"], pred["T_0.8m_C"], pred["T_1m_C"]],
    color="crimson", zorder=5, label="Sampled points",
)
# Mark the 5 Excel measurement temperatures on the profile

ax.axhline(T_heat_C, color="gray", ls="--", alpha=0.6, label=f"Heat medium ({T_heat_C:.0f} °C)")
# Horizontal reference line at heating medium temperature

ax.set_xlabel("Reactor length (m)")
# X-axis label: axial position along bed

ax.set_ylabel("Temperature (°C)")
# Y-axis label: gas temperature

ax.set_title("Axial temperature profile — CH3OH -> CO + 2H2")
# Plot title showing reaction pathway

ax.legend()
# Show legend for model curve, points, and reference line

ax.grid(True, alpha=0.3)
# Add light grid for readability

plt.tight_layout()
# Adjust layout to prevent label clipping

plt.show()
# Display the temperature profile figure

## 8. Compare with Methanol Reforming Data.xlsx (inputs only)

In [ ]:
# Run physics model on every data row (inputs only) and compare to data outputs
rows = []
n_fail = 0
first_error = None

print(f"Comparing physics model vs Excel on {len(df)} cases...")
for i, row in df.iterrows():
    rec = {
        "T_in_C": float(row["T_in_C"]),
        "P_in_bar_g": float(row["P_in_bar_g"]),
        "T_heat_C": float(row["T_heat_C"]),
        "F_in_kgmole_h": float(row["F_in_kgmole_h"]),
    }
    # Store Excel reference outputs
    for c in OUTPUT_COLS:
        rec[f"{c}_excel"] = float(row[c]) if pd.notna(row[c]) else float("nan")

    try:
        pred = predict_outputs(
            rec["T_in_C"], rec["P_in_bar_g"], rec["T_heat_C"], rec["F_in_kgmole_h"]
        )
        for c in OUTPUT_COLS:
            rec[f"{c}_pred"] = float(pred[c])
        rec["ok"] = True
    except Exception as exc:
        n_fail += 1
        if first_error is None:
            first_error = exc
        for c in OUTPUT_COLS:
            rec[f"{c}_pred"] = float("nan")
        rec["ok"] = False

    rows.append(rec)
    if (len(rows) % 100) == 0:
        print(f"  {len(rows)}/{len(df)}")

results = pd.DataFrame(rows)
results_ok = results.loc[results["ok"]].reset_index(drop=True)
print(f"Successful cases: {len(results_ok)} / {len(results)} (failed={n_fail})")
if first_error is not None and len(results_ok) == 0:
    raise RuntimeError(f"All physics comparisons failed. First error: {first_error!r}")
if first_error is not None:
    print(f"  First failure: {first_error!r}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# Create figure with 2 side-by-side subplots for validation charts

axes[0].scatter(results_ok["X_overall_pct_excel"], results_ok["X_overall_pct_pred"], alpha=0.7)
# Plot Excel vs model conversion for all successful cases

lim = max(
    results_ok["X_overall_pct_excel"].max(),
    results_ok["X_overall_pct_pred"].max(),
    1e-6,
) * 1.1
# Set axis limit slightly above max value for padding

axes[0].plot([0, lim], [0, lim], "k--", lw=1)
# Draw perfect-match diagonal line (y = x)

axes[0].set_xlabel("Data conversion (%)")
# Label x-axis for conversion plot

axes[0].set_ylabel("Model conversion (%)")
# Label y-axis for conversion plot

axes[0].set_title("Overall conversion")
# Title for conversion comparison plot

axes[0].grid(True, alpha=0.3)
# Add grid to conversion plot

axes[1].scatter(results_ok["T_1m_C_excel"], results_ok["T_1m_C_pred"], alpha=0.7)
# Plot Excel vs model outlet temperature

t0 = min(results_ok["T_1m_C_excel"].min(), results_ok["T_1m_C_pred"].min()) - 5
# Lower bound for temperature axis with margin

t1 = max(results_ok["T_1m_C_excel"].max(), results_ok["T_1m_C_pred"].max()) + 5
# Upper bound for temperature axis with margin

axes[1].plot([t0, t1], [t0, t1], "k--", lw=1)
# Draw perfect-match diagonal for temperature

axes[1].set_xlabel("Data Temp_1m (°C)")
# Label x-axis for temperature plot

axes[1].set_ylabel("Model Temp_1m (°C)")
# Label y-axis for temperature plot

axes[1].set_title("Outlet temperature")
# Title for outlet temperature comparison

axes[1].grid(True, alpha=0.3)
# Add grid to temperature plot

fig.suptitle("Physics model vs Excel (Excel data)")
plt.tight_layout()
# Adjust subplot spacing

plt.show()
# Display both validation plots

r0 = results_ok.iloc[0]
# Pick first successful row for detailed error table

compare = pd.DataFrame({
    "Output": OUTPUT_COLS,
    "Excel": [r0[f"{c}_excel"] for c in OUTPUT_COLS],
    "Physics model": [r0[f"{c}_pred"] for c in OUTPUT_COLS],
})
# Build table comparing all 8 outputs for one case

compare["Abs error"] = (compare["Physics model"] - compare["Excel"]).abs()
# Compute absolute difference between model and Excel for each output

print("Detailed comparison — first successful data row:")
display(compare.round(4))
# Show comparison table rounded to 4 decimals
